# Nettoyage des données pas utile

In [ ]:
# ----------------------
# 2. Nettoyage des variables non pertinentes ou redondantes
# ----------------------

# ----------------------
# Feature Engineering manuel : création de nouvelles colonnes
# ----------------------

# Total des salles de bain (en comptant les demi-salles)
data_raw['TotalBathrooms'] = (
    data_raw.get('FullBath', 0) +
    0.5 * data_raw.get('HalfBath', 0) +
    data_raw.get('BsmtFullBath', 0) +
    0.5 * data_raw.get('BsmtHalfBath', 0)
)

# Surface totale habitable (sous-sol + RDC + étage)
data_raw['TotalSF'] = (
    data_raw.get('TotalBsmtSF', 0) +
    data_raw.get('1stFlrSF', 0) +
    data_raw.get('2ndFlrSF', 0)
)

# Surface totale de porche
data_raw['TotalPorchSF'] = (
    data_raw.get('OpenPorchSF', 0) +
    data_raw.get('EnclosedPorch', 0) +
    data_raw.get('3SsnPorch', 0) +
    data_raw.get('ScreenPorch', 0)
)

# Age de la maison à la vente
if 'YrSold' in data_raw and 'YearBuilt' in data_raw:
    data_raw['HouseAge'] = data_raw['YrSold'] - data_raw['YearBuilt']

# Score global qualité * condition
data_raw['OverallGrade'] = (
    data_raw.get('OverallQual', 0) * data_raw.get('OverallCond', 0)
)

# Nombre de pièces par salle de bain (indicateur de confort)
if 'TotRmsAbvGrd' in data_raw:
    data_raw['BathPerRoom'] = data_raw['TotalBathrooms'] / data_raw['TotRmsAbvGrd'].replace(0, np.nan)

# Volume de garage (voitures * surface)
data_raw['GarageVolume'] = (
    data_raw.get('GarageCars', 0) * data_raw.get('GarageArea', 0)
)

quantitative += ['TotalBathrooms', 'TotalSF', 'TotalPorchSF', 'HouseAge',
                 'OverallGrade', 'BathPerRoom', 'GarageVolume']

# Variables à retirer pour faible contribution ou multi-colinéarité
vars_to_drop = [
    'Street', 'Alley', 'LandSlope', 'Utilities', 'RoofMatl',
    'Condition2', 'Heating', 'Functional', 'LowQualFinSF',
    '3SsnPorch', 'PoolArea', 'PoolQC', 'MiscVal', 'MoSold',
    'BsmtHalfBath', 'BsmtFinSF2', 'EnclosedPorch', 'GarageArea',
    'GarageCars', 'OpenPorchSF', 'ScreenPorch', 'TotalBsmtSF',
    '1stFlrSF', '2ndFlrSF', 'FullBath', 'HalfBath', 'BsmtFullBath'
]

# Nettoyage des listes
qualitative = [col for col in qualitative if col not in vars_to_drop]
quantitative = [col for col in quantitative if col not in vars_to_drop]
date = [col for col in date if col not in vars_to_drop]

# Modèle 1 - Random Forest (Brian)

In [ ]:
# ----------------------
# 1. Modèle avec paramètres fixes
# ----------------------
random_forest_model = RandomForestRegressor(random_state=42, n_jobs=-1)

# ----------------------
# 2. Méthode d'optimisation des hyperparamètres
# ----------------------
def objective(trial):
    
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'max_depth': trial.suggest_int('max_depth', 5, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }

    random_forest_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('random_forest_model', random_forest_model.set_params(**params))
    ])

    cv_score = cross_val_score(random_forest_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1).mean()

    return -cv_score

# ----------------------
# 3. Lancer l'optimisation avec Optuna
# ----------------------
study = optuna.create_study(direction='minimize', study_name='random_forest_study') 
study.optimize(objective, n_trials=100)

# ----------------------
# 4. Afficher les meilleurs paramètres et le score
# ----------------------
print("--------------------------------------")
print("Meilleurs paramètres =", study.best_params)
print("Score RMSE (CV) :", study.best_value)

In [ ]:
# ----------------------
# 1. Affecter les meilleurs paramètres au modèle RandomForest
# ----------------------
random_forest_model.set_params(**study.best_params)

# ----------------------
# 2. Pipeline complète avec le modèle RandomForest
# ----------------------
random_forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), 
    ('regressor', random_forest_model)
])

# ----------------------
# 3. Évaluation avec cross-validation
# ----------------------
cv_score = cross_val_score(
    random_forest_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
).mean()

print("RMSE moyen :", -cv_score)

# Modèle 2 - Régression Lasso (Brian)

In [ ]:
# ----------------------
# 1. Modèle Lasso avec paramètres fixes
# ----------------------
lasso_model = Lasso(random_state=42, max_iter=10000)

# ----------------------
# 2. Méthode d'optimisation des hyperparamètres
# ----------------------
def objective(trial):
    alpha = trial.suggest_float('alpha', 1e-4, 10.0, log=True)

    lasso_model.set_params(alpha=alpha)

    lasso_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor), 
        ('regressor', lasso_model)
    ])

    cv_score = cross_val_score(lasso_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1).mean()

    return -cv_score

# ----------------------
# 3. Lancer l'optimisation avec Optuna
# ----------------------
study = optuna.create_study(direction='minimize', study_name='lasso_study')
study.optimize(objective, n_trials=100)

# ----------------------
# 4. Afficher les meilleurs paramètres et le score
# ----------------------
print("--------------------------------------")
print("Meilleur alpha :", study.best_params['alpha'])
print("Score RMSE (CV) :", study.best_value)

In [ ]:
# ----------------------
# 1. Pipeline complète avec les meilleurs paramètres Lasso
# ----------------------
lasso_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Lasso(alpha=study.best_params['alpha'], random_state=42, max_iter=10000))
])

# ----------------------
# 2. Évaluation avec cross-validation
# ----------------------
cv_score = cross_val_score(
    lasso_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
).mean()

print("RMSE moyen :", -cv_score)

# Decision Tree

Nous avons utilisé cv=15 pendant le tuning car c'est un modèle rapide à entraîner, ce qui permettait une estimation plus robuste et réduit la variance entre les trials d’Optuna. 

Afin de garantir une évaluation cohérente et comparable entre tous les modèles, nous avons fixé cv=10 pour la validation finale. Cette valeur représente un bon compromis entre fiabilité statistique et coût de calcul

## Modèle Stacking Regressor

In [ ]:
def stacking_objective(trial, preprocessing_pipeline, X, y, cv, scoring):
    lasso_pipeline = Pipeline([
        ('preprocessing', preprocessing_pipeline),
        ('model', Lasso(lasso_best_params['alpha'], max_iter=10000, random_state=42))
    ])

    random_forest_pipeline = Pipeline([
        ('preprocessing', preprocessing_pipeline),
        ('model', RandomForestRegressor(
            n_estimators=random_forest_best_params['n_estimators'],
            max_depth=random_forest_best_params['max_depth'],
            min_samples_split=random_forest_best_params['min_samples_split'],
            min_samples_leaf=random_forest_best_params['min_samples_leaf'],
            max_features= random_forest_best_params['max_features'],
            random_state=42,
            n_jobs=-1
        ))

    ])

    xgboost_pipeline = Pipeline([
        ('preprocessing', preprocessing_pipeline),
        ('model', XGBRegressor(
            eta=xgboost_best_params['eta'],
            n_estimators=xgboost_best_params['n_estimators'],
            max_depth=xgboost_best_params['max_depth'],
            min_child_weight=xgboost_best_params['min_child_weight'],
            subsample=xgboost_best_params['subsample'],
            colsample_bytree=xgboost_best_params['colsample_bytree'],
            gamma=xgboost_best_params['gamma'],
            alpha=xgboost_best_params['alpha'],
            lambda_=xgboost_best_params['lambda'],
            objective='reg:squarederror',
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Modifier les paramètres du Ridge ? (Alpha)
    final_estimator = Ridge(alpha=trial.suggest_float('ridge_alpha', 0.01, 10.0, log=True))

    stacking = StackingRegressor(
        estimators=[
            ('lasso', lasso_pipeline),
            ('random_forest', random_forest_pipeline),
            ('xgboost', xgboost_pipeline)
        ],
        final_estimator=final_estimator,
        cv=cv,
        n_jobs=-1
    )

    scores = cross_val_score(stacking, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    return -scores.mean()

In [ ]:
model_configs = {
    'Stacking': stacking_objective
}
results = evaluate_and_optimize_models(model_configs, X_train, y_train, preprocessing_pipeline, n_trials=5)

In [ ]:
stacking_study = results['Stacking'][2]
plot_slice(stacking_study).show()

In [ ]:
stacking_best_params = results['Stacking'][1]

model = Ridge(alpha=stacking_best_params['ridge_alpha'])

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('stacking', model)
    ])

model_pipeline.fit(X_train, y)

y_pred = model_pipeline.predict(X_test)

kaggle_submission = pd.DataFrame()
kaggle_submission['Id'] = X_test.index
kaggle_submission['SalePrice'] = np.expm1(y_pred)
kaggle_submission.to_csv('Data/stacking.csv', index=False)